## Sentiment Analysis Methodology
 
The dataset where the models were tested is available here: https://huggingface.co/datasets/ElKulako/stocktwits-crypto <br>
It contains all cryptocurrency-related posts from the StockTwits website (around 1.3M), from 1st of November 2021 to the 15th of June 2022.
 
The provided data was already cleaned and preprocessed, in particular it was removed:
 
- cashtags, hashtags, usernames,
- URLs, crypto wallets,
- Chinese, Korean and Japanese characters,
- (most) UTF-8 encoding issues
- All posts shorter than 4 words
- All duplicate posts
- Spacing and punctuation issues, converted all text to lowercase

Initially, I intended to test on the dataset the following models:
 
- Vader: Lexicon and rule-based sentiment analysis tool that is specifically attuned to sentiments expressed in social media
- Support Vector Machines: A traditional, statistical classifier.
- Random Forest: An ensemble machine learning method utilizing deep decision trees.
- Gemma 2B: Google's lightweight, 2-billion parameter Large Language Model.

All this models were to be benchmarked against CryptoBERT, a specialized Transformer machine learning model trained on 3.2 millions social media posts of various cryptocurrencies. <br>
The benchmark results for CryptoBERT are publicly available, making it an ideal competitor, especially since it was explicitly fine-tuned on a portion of this exact StockTwits dataset.
 
After some testing I decided to drop Gemma 2B and Random forest for two main reasons:
 
- Performance: Preliminary testing showed they did not maintain competitive accuracy compared to the SVM and CryptoBERT.
- Computational Efficiency: At scale the inference time for the LLM and the training complexity of the Random Forest were prohibitively high.

Consequently, the final evaluation focuses on a comparison between VADER, SVM and CryptoBERT.
 
Results & Evaluation
 
VADER and SVM were evaluated on the same exact 83k rows test set of the CryptoBERT paper
 
VADER: Accuracy: 37.34% | Macro F1: 0.3679
 
As expected, the traditional rule-based lexicon struggled. <br>
VADER's reliance on standard English definitions proved inadequate for capturing the nuanced, slang-heavy and emoji-driven nature of cryptocurrency posts.
 
CryptoBERT: Accuracy: 55.60% | Macro F1: 0.5579
 
The specialized transformer established a balanced standard. <br>
Thanks to its deep contextual embeddings pre-trained on crypto-specific text, it successfully managed the complex class imbalances and set a high benchmark for the Macro F1 score.
 
SVM: Accuracy: 59.71% | Macro F1: 0.5916
 
The SVM performed exceptionally well, outperforming CryptoBERT in both categories. <br>
By utilizing a wide range of n-grams (1–5), the model successfully captured the specific "power phrases" and slang that drive sentiment on StockTwits. <br>
It achieved an accuracy 4.11% higher than the state-of-the-art transformer.
 
Conclusion

The results demonstrate that a properly tuned statistical machine learning pipeline can surpass a specialized Transformer model in both raw accuracy and Macro F1 score on this dataset. <br>
The SVM’s superior performance, combined with its significantly lower computational overhead, makes it the most viable and scalable solution for high-frequency sentiment analysis in the cryptocurrency domain.

In [ ]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import pickle

TARGET_NAMES = ["Bearish", "Neutral", "Bullish"]
LABEL_ORDER  = [0, 1, 2]

vader = SentimentIntensityAnalyzer()

def get_vader_sentiment(text: str) -> int:
    compound = vader.polarity_scores(text)["compound"]
    if compound >= 0.05:
        return 2   # Bullish
    elif compound <= -0.05:
        return 0   # Bearish
    else:
        return 1   # Neutral

def evaluate_model(model_name: str, y_true, y_pred):
    
    print(f" {model_name}")
    print(f"  Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(classification_report(y_true, y_pred, labels=LABEL_ORDER, target_names=TARGET_NAMES, digits=4))

if __name__ == "__main__":

    df_train_1 = pd.read_csv("API_App/data/sentiment/stockwits_train_1.csv")
    df_train_2 = pd.read_csv("API_App/data/sentiment/stockwits_train_2.csv")
    df_test = pd.read_csv("API_App/data/sentiment/stockwits_test.csv")

    df_train = pd.concat([df_train_1, df_train_2], ignore_index=True)
    df_tune = df_train.sample(frac=0.2, random_state=42)
    x_tune, y_tune = df_tune["text"].tolist(), df_tune["label"].tolist()
    
    x_train, y_train = df_train["text"].tolist(), df_train["label"].tolist()
    x_test, y_test = df_test["text"].tolist(), df_test["label"].tolist()
    
    results_df = pd.DataFrame({"text": x_test, "label": y_test})

    print("Running VADER...")
    results_df["vader_pred"] = results_df["text"].apply(get_vader_sentiment)
    
    print("Training & running SVM...")
    svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LinearSVC(class_weight="balanced", dual=False)),
    ])

    param_grid = {
    "tfidf__ngram_range": [(1,4),(1, 5)], 
    "tfidf__min_df": [1, 2],                       
    "tfidf__max_df": [0.9, 0.95, 1.0],                     
    "tfidf__sublinear_tf": [False],                                
}
    
    print("Running Grid Search (this will take a while)...")
    grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    cv=3,          
    scoring='f1_macro', 
    n_jobs=2,  
    verbose=3       
)
    grid_search.fit(x_tune, y_tune)
    print("\nBest Parameters found:", grid_search.best_params_)
    
    best_svm = grid_search.best_estimator_
    best_svm.fit(x_train, y_train)          
    results_df["svm_pred"] = best_svm.predict(x_test)

    evaluate_model("VADER", results_df["label"], results_df["vader_pred"])
    evaluate_model("SVM", results_df["label"], results_df["svm_pred"])
    
    print(f"CryptoBert")
    print(f"  Accuracy: 0.5560")
    print(f"  Macro AVG F1 Score: 0.5579")
    print(f"  Macro AVG Precision: 0.4658")
    print(f"  Macro AVG Recall: 0.6044")

    pickle.dump(best_svm, open("../data/models/svm.pkl", "wb"))

Running VADER...
Training & running SVM...
Running Grid Search (this will take a while)...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV 1/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 4), tfidf__sublinear_tf=False;, score=0.519 total time= 1.6min
[CV 2/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 4), tfidf__sublinear_tf=False;, score=0.520 total time= 1.7min
[CV 3/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 4), tfidf__sublinear_tf=False;, score=0.519 total time= 1.6min
[CV 1/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 5), tfidf__sublinear_tf=False;, score=0.520 total time= 1.9min
[CV 2/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 5), tfidf__sublinear_tf=False;, score=0.521 total time= 1.9min
[CV 3/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 5), tfidf__sublinear_tf=False;, score=0.519 total time= 2.2min
[CV 1/3] END tfidf__max_df=0.9, tfidf_

# Building the training dataset

Now that we have identified the SVM model as our candidate for sentiment analysis we must produce a dataset where our classifiers can be trained and tested on. <br>
To do this, we leverage a kaggle dataset containing all reddit posts of the whole 2022 year: https://www.kaggle.com/datasets/leukipp/reddit-crypto-data

Since this contains the posts of dozens of subreddits, we limit ourselves to the following ones:

- Bitcoin (https://www.reddit.com/r/Bitcoin/)
- Solana (https://www.reddit.com/r/solana/)
- Ethereum (https://www.reddit.com/r/ethereum/)
- Cryptocurrencies (https://www.reddit.com/r/CryptoCurrencies/)
- CryptoCurrency (https://www.reddit.com/r/CryptoCurrency/)
- Crypto_currency_news (https://www.reddit.com/r/Crypto_Currency_News/)

The first 3 are coin-specific while the last 3 are more "general" crypto oriented.

## Dataset: Sentiment columns

From the original reddit dataset we are mainly interested in two columns, the "text" column which combines title and body and the "created" column which tells us the publication time. <br>
Out of these two fields we generate many new features, including:

### 1. Base Sentiment Classifications
- **`sentiment_[btc/eth/sol/gen]`**: The classification generated by passing the cleaned `"text"` column through our trained SVM model. 
  - `0` = Bearish
  - `1` = Neutral
  - `2` = Bullish

### 2. Aggregated Hourly Ratios & Scores (Resampled to 60-minute windows)
- **`[Coin]_bullish_ratio` / `[Coin]_bearish_ratio`**: The proportion of all community discussion during that hour holding a specific outlook. <br> 
Calculated as the total number of bullish (2) or bearish (0) posts divided by the `total_post_count`.
- **`[Coin]_sentiment_score`**: The net sentiment relative only to the posts that actually mentioned that specific Coin, balancing out the neutral noise.
- **`[Coin]_weighted_sentiment`**: The Coin's core sentiment score scaled by the overall hourly post volume, capturing the total intensity of the conversation.

### 3. Sentiment Momentum & Polarity
- **`[Coin]_delta_sentiment`**: The hourly change in the Coin's sentiment score compared to the previous hour, highlighting sudden shifts in market mood.
- **`[Coin]_polarity`**: The raw difference between the bullish ratio and bearish ratio, looking for the bias of the community before it is converted into lagged metrics.

### 4. Community Volume Trajectory Metrics
- **`post_volume_24h_avg`**: A 24-hour rolling average of `total_post_count` which establishes a baseline of typical community activity over a full day.
- **`post_spike_ratio`**: A normalized activity metric calculated as:
  $$\frac{\text{Current Hourly Post Count}}{\text{24h Rolling Average Post Count} + 1}$$
  This captures sudden anomalies or massive spikes in discussion relative to recent trends, serving as an indicator for breaking news or panic.

### 5. Cyclical Time Features
- **`hour_sin` / `hour_cos`**: Generated by converting the `"created"` timestamp into the hour of the day (0–23) and transforming it using sine and cosine functions.
- **`day_sin` / `day_cos`**: The same cyclical transformation applied to the day of the week (0–6) to map weekly patterns smoothly.

### 6. Multi-Interval Lagged Features
- **`[Coin]_polarity_lag_[1/3/6/12]` & `[Coin]_delta_sentiment_lag_[1/3/6/12]`**: Historical snapshots of sentiment polarity and momentum shifted backwards by 1, 3, 6, and 12 hours. These allow the classifiers to look back in time and recognize trends or delayed community reactions.

## Dataset: Financial columns

The market data is ingested hourly using CCXT feeds for `BTC/USDT`, `ETH/USDT`, and `SOL/USDT`. <br>
This raw pricing data is processed into returns, rolling standard deviations, and log-transformed metrics to track market momentum and risk.

### 1. Base Market Features (Calculated for BTC-USD, ETH-USD, and SOL-USD)
- **`[Ticker]_return`**: The hourly percentage change in price, capturing short-term market direction and price momentum.
- **`[Ticker]_volatility`**: The rolling standard deviation of asset returns. This measures the raw dispersion of prices over a given window and serves as the continuous baseline for our target definitions.
- **`[Ticker]_vol_log`**: The natural logarithm of the volatility metric ($\ln(\text{volatility})$).

### 2. Multi-Interval Financial Lags
- **`[Ticker]_return_lag_[1/3/6/12]`**: Historical asset returns shifted backward by 1, 3, 6, and 12 hours. <br>
These features allow the model to recognize traditional price momentum trends
- **`[Ticker]_volatility_lag_[1/3/6/12]`**: Past volatility signatures shifted backward, as past volatility can be an excellent predictor of future volatility.
- **`[Ticker]_vol_log_lag_[1/3/6/12]`**: Normalized historical volatility

## Inherent Limitations & Market Regime Risks

A significant limitation of this dataset and the resulting predictive models derives from the temporal scope of the training data, which is strictly bounded within the calendar year **2022**. 

### 1. 2022 Market Regime Overfitting
The year 2022 represented a historic macroeconomic shift and an aggressive bear market for digital assets, frequently referred to as the "Crypto Winter." 
* **Skewed Returns and Volatility:** The structural pricing data (`[Ticker]_return`) and volatility baselines (`[Ticker]_volatility`) are highly reflective of a persistent downward trend punctuated by systemic, high-leverage market collapses (e.g., the liquidations of Three Arrows Capital, Celsius, and the collapse of FTX). 
* **Target Boundary Distortion:** Because our categorical targets (`BTC_vol_target`, `ETH_vol_target`, `SOL_vol_target`) are generated using quantiles (`pd.qcut`) derived strictly from 2022 data, the mathematical boundaries for what constitutes "Low," "Medium," and "High" volatility are inherently calibrated to an exceptionally high-volatility regime. 

### 2. Generalization Challenges
Because of this training window, models trained on this dataset will likely suffer from **concept drift** when deployed or evaluated on data from different market cycles (eg. prolonged bull market or a low volatility phase). 
* In a calmer market, a normal day's volatility might be universally categorized as "Low Volatility (Class 0)" by these 2022-calibrated boundaries.
* Similarly, the sentiment-to-price relationships captured during panic cycles may not map correctly to environments driven by "FOMO" or institutional accumulation.

In [ ]:
import pandas as pd
import re
import pickle
import ccxt
import numpy as np

def clean_reddit_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace('&#x200B;', '')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def prepare_market_data(df, ticker):
    df = df.rename(columns={
        "Close": f"{ticker}_close",
        "Volume": f"{ticker}_volume"
    })
    
    df[f"{ticker}_return"] = df[f"{ticker}_close"].pct_change()
    df[f"{ticker}_volatility"] = (df["High"] - df["Low"]) / df["Open"]
    df[f"{ticker}_vol_log"] = np.log1p(df[f"{ticker}_volume"])
    
    return df[[f"{ticker}_close", f"{ticker}_return", f"{ticker}_volatility", f"{ticker}_vol_log"]]

def fetch_ccxt(symbol, start_str, end_str, timeframe='1h'):

    exchange = ccxt.binance({'enableRateLimit': True})
    
    start = exchange.parse8601(f"{start_str}T00:00:00Z")
    end = exchange.parse8601(f"{end_str}T23:59:59Z")
    
    all_kline = []

    if start is None or end is None:
        raise ValueError("Could not parse the provided start_str or end_str dates.")
    
    while start < end:
        
        try:
            klines = exchange.fetch_ohlcv(symbol, timeframe, start, limit=1000)
            if not klines:
                break
                
            start = klines[-1][0] + 1 
            all_kline.extend(klines)
            
            if klines[-1][0] >= end:
                break
                
        except Exception as e:
            print(f"Error fetching {symbol}: {e}")
            break
            
    df = pd.DataFrame(all_kline, columns=['Datetime', 'Open', 'High', 'Low', 'Close', 'Volume'])
    
    df['Datetime'] = pd.to_datetime(df['Datetime'], unit='ms', utc=True)
    df.set_index('Datetime', inplace=True)
    return df

def generate_financial_columns(group):
    post_count = len(group)
    btc_count = group["sentiment_btc"].notna().sum()
    eth_count = group["sentiment_eth"].notna().sum()
    sol_count = group["sentiment_sol"].notna().sum()
    gen_count = group["sentiment_gen"].notna().sum()
    bullish_btc = (group["sentiment_btc"] == 2).sum()
    bearish_btc = (group["sentiment_btc"] == 0).sum()
    bullish_eth = (group["sentiment_eth"] == 2).sum()
    bearish_eth = (group["sentiment_eth"] == 0).sum()
    bullish_sol = (group["sentiment_sol"] == 2).sum()
    bearish_sol = (group["sentiment_sol"] == 0).sum()
    bullish_gen = (group["sentiment_gen"] == 2).sum()
    bearish_gen = (group["sentiment_gen"] == 0).sum()

    return pd.Series({
        "total_post_count": post_count,
        "BTC_bullish_ratio": bullish_btc / post_count if post_count else 0,
        "ETH_bullish_ratio": bullish_eth / post_count if post_count else 0,
        "SOL_bullish_ratio": bullish_sol / post_count if post_count else 0,
        "GEN_bullish_ratio": bullish_gen / post_count if post_count else 0,
        "BTC_bearish_ratio": bearish_btc / post_count if post_count else 0,
        "ETH_bearish_ratio": bearish_eth / post_count if post_count else 0,
        "SOL_bearish_ratio": bearish_sol / post_count if post_count else 0,
        "GEN_bearish_ratio": bearish_gen / post_count if post_count else 0,
        "BTC_sentiment_score": (bullish_btc - bearish_btc) / btc_count if btc_count else 0,
        "ETH_sentiment_score": (bullish_eth - bearish_eth) / eth_count if eth_count else 0,
        "SOL_sentiment_score": (bullish_sol - bearish_sol) / sol_count if sol_count else 0,
        "GEN_sentiment_score": (bullish_gen - bearish_gen) / gen_count if gen_count else 0,
        "BTC_weighted_sentiment": ((bullish_btc - bearish_btc) / btc_count) * post_count if btc_count else 0,
        "ETH_weighted_sentiment": ((bullish_eth - bearish_eth) / eth_count) * post_count if eth_count else 0,
        "SOL_weighted_sentiment": ((bullish_sol - bearish_sol) / sol_count) * post_count if sol_count else 0,
        "GEN_weighted_sentiment": ((bullish_gen - bearish_gen) / gen_count) * post_count if gen_count else 0
    })
    

def clean_dataset(df_btc, df_eth, df_sol) -> None:
    
    datasets = ["bitcoin", "cryptocurrency", "cryptocurrencies", "crypto_currency_news", "ethereum", "solana"]
    prefixes = ['BTC', 'ETH', 'SOL', 'GEN']
    tickers = ['BTC-USD', 'ETH-USD', 'SOL-USD']
    all_dfs = []
    svm_model = pickle.load(open("data/models/svm.pkl", "rb"))
    for dataset in datasets:

        df = pd.read_csv(f"data/raw_reddit/{dataset}/submission.csv")

        df = df[(df["removed"] == 0) & (df["deleted"] == 0)]
        df = df.drop(columns=[
        "submission",             
        "author",              
        "retrieved",      
        "edited",    
        "pinned",     
        "archived",     
        "locked",       
        "removed",         
        "deleted",           
        "is_video",                
        "is_original_content",     
        "gilded",                  
        "total_awards_received",  
        "num_crossposts",          
        "thumbnail",               
        "shortlink"                
    ])
        df = df.dropna(subset=["created", "title"])
        df["text"] = (df["title"].fillna("") + " " + df["selftext"].fillna("")).str.strip()
        df.drop(columns=["title", "selftext"], inplace=True)
        
        df["text"] = df["text"].apply(clean_reddit_text)
        df["created"] = pd.to_datetime(df["created"], unit="s", utc=True)
        
        if dataset == "bitcoin":
            df["sentiment_btc"] = svm_model.predict(df["text"])
        elif dataset == "ethereum":
            df["sentiment_eth"] = svm_model.predict(df["text"])
        elif dataset == "solana":
            df["sentiment_sol"] = svm_model.predict(df["text"])
        else:
            df["sentiment_gen"] = svm_model.predict(df["text"])

        all_dfs.append(df)

    df_final = pd.concat(all_dfs, ignore_index=True)
    df_final = df_final.set_index("created").sort_index()

    df_final = df_final.resample("60min").apply(generate_financial_columns)
    df_final["BTC_delta_sentiment"] = df_final["BTC_sentiment_score"].diff().fillna(0)
    df_final["ETH_delta_sentiment"] = df_final["ETH_sentiment_score"].diff().fillna(0)
    df_final["SOL_delta_sentiment"] = df_final["SOL_sentiment_score"].diff().fillna(0)
    df_final["GEN_delta_sentiment"] = df_final["GEN_sentiment_score"].diff().fillna(0)
    df_final = df_final.join([df_btc, df_eth, df_sol], how="inner")
    
    df_final['hour_of_day'] = df_final.index.hour
    df_final['hour_sin'] = np.sin(2 * np.pi * df_final['hour_of_day'] / 24)
    df_final['hour_cos'] = np.cos(2 * np.pi * df_final['hour_of_day'] / 24)     
    
    df_final['day_of_week'] = df_final.index.dayofweek
    df_final['day_sin'] = np.sin(2 * np.pi * df_final['day_of_week'] / 7)
    df_final['day_cos'] = np.cos(2 * np.pi * df_final['day_of_week'] / 7)   
    
    df_final.drop(columns=['hour_of_day', 'day_of_week'], inplace=True)
    
    for p in prefixes:
        df_final[f'{p}_polarity'] = df_final[f'{p}_bullish_ratio'] - df_final[f'{p}_bearish_ratio']

    features_to_lag = []
    
    for p in prefixes:
        features_to_lag.append(f'{p}_polarity')
        features_to_lag.append(f'{p}_delta_sentiment')

    for t in tickers:
        features_to_lag.append(f'{t}_return')
        features_to_lag.append(f'{t}_volatility')
        features_to_lag.append(f'{t}_vol_log')
        
    for col in features_to_lag:
        df_final[f'{col}_lag_1'] = df_final[col].shift(1)
        df_final[f'{col}_lag_3'] = df_final[col].shift(3)
        df_final[f'{col}_lag_6'] = df_final[col].shift(6)
        df_final[f'{col}_lag_12'] = df_final[col].shift(12)
        
    df_final.drop(columns=[f'{p}_polarity' for p in prefixes], inplace=True)
    
    train_size = int(len(df_final) * 0.7)
    train_df = df_final.iloc[:train_size].copy()

    _, btc_bins = pd.qcut(train_df['BTC-USD_volatility'], q=3, retbins=True)
    _, eth_bins = pd.qcut(train_df['ETH-USD_volatility'], q=3, retbins=True)
    _, sol_bins = pd.qcut(train_df['SOL-USD_volatility'], q=3, retbins=True)

    for b in [btc_bins, eth_bins, sol_bins]:
        b[0] = -np.inf
        b[-1] = np.inf

    df_final['BTC_vol_target'] = pd.cut(df_final['BTC-USD_volatility'], bins=btc_bins, labels=[0, 1, 2])
    df_final['ETH_vol_target'] = pd.cut(df_final['ETH-USD_volatility'], bins=eth_bins, labels=[0, 1, 2])
    df_final['SOL_vol_target'] = pd.cut(df_final['SOL-USD_volatility'], bins=sol_bins, labels=[0, 1, 2])

    print("BTC: MODEL BOUNDARIES")
    print(f"Low Volatility (Class 0): Everything from 0% up to {btc_bins[1]*100:.3f}%")
    print(f"Medium Volatility (Class 1): Between {btc_bins[1]*100:.3f}% and {btc_bins[2]*100:.3f}%")
    print(f"High Volatility (Class 2): Everything above {btc_bins[2]*100:.3f}%")

    print("ETH: MODEL BOUNDARIES")
    print(f"Low Volatility (Class 0): Everything from 0% up to {eth_bins[1]*100:.3f}%")
    print(f"Medium Volatility (Class 1): Between {eth_bins[1]*100:.3f}% and {eth_bins[2]*100:.3f}%")
    print(f"High Volatility (Class 2): Everything above {eth_bins[2]*100:.3f}%")

    print("SOL: MODEL BOUNDARIES")
    print(f"Low Volatility (Class 0): Everything from 0% up to {sol_bins[1]*100:.3f}%")
    print(f"Medium Volatility (Class 1): Between {sol_bins[1]*100:.3f}% and {sol_bins[2]*100:.3f}%")
    print(f"High Volatility (Class 2): Everything above {sol_bins[2]*100:.3f}%")

    df_final['BTC_vol_target'] = df_final['BTC_vol_target'].shift(-1)
    df_final['SOL_vol_target'] = df_final['SOL_vol_target'].shift(-1)
    df_final['ETH_vol_target'] = df_final['ETH_vol_target'].shift(-1)
    
    df_final['post_volume_24h_avg'] = df_final['total_post_count'].rolling(window=24).mean()
    df_final['post_spike_ratio'] = df_final['total_post_count'] / (df_final['post_volume_24h_avg'] + 1)
    
    lag_cols = [f'{col}_lag_{lag}' for col in features_to_lag for lag in [1, 3, 6, 12]]
    df_final = df_final.dropna(subset=['BTC_vol_target', 'SOL_vol_target', 'ETH_vol_target', 'post_volume_24h_avg'] + lag_cols)
    df_final = df_final.reset_index()
      
    df_final.to_csv("API_App/data/clean_reddit/final_submission_clean.csv", index=False)
        
if __name__ == "__main__":

    btc_raw = fetch_ccxt("BTC/USDT", "2022-01-01", "2022-12-31")
    eth_raw = fetch_ccxt("ETH/USDT", "2022-01-01", "2022-12-31")
    sol_raw = fetch_ccxt("SOL/USDT", "2022-01-01", "2022-12-31")

    btc = prepare_market_data(btc_raw, "BTC-USD")
    eth = prepare_market_data(eth_raw, "ETH-USD")
    sol = prepare_market_data(sol_raw, "SOL-USD")

    clean_dataset(btc, eth, sol)



# Model Training & Volatility Forecasting Architecture

The primary objective of this phase is to build a "Warning system" capable of predicting next-hour Bitcoin market volatility. <br>
Rather than predicting price direction, our models isolate and forecast market explosiveness.

Successfully identifying these high-volatility regimes ahead of time allows traders to deploy direction-neutral options strategies, such as a **Long Straddle** or **Long Strangle**, which profit off extreme price expansion regardless of whether the market moves up or down.

## 1. Experimental Methodology & Validation Setup

To simulate a true production environment and prevent severe data leakage, the evaluation pipeline strictly relies on a **Time Series Split (`TimeSeriesSplit(n_splits=5)`)**.

Our training function evaluates three model across chronological folds:
- **XGBoost (Extreme Gradient Boosting):** Optimized with `multi:softprob` to handle complex, non-linear interactions across sequential tabular data.
- **Random Forest:** Employed as a robust bagging baseline, utilizing `class_weight="balanced"` to counteract sample size imbalances.
- **LightGBM (Light Gradient Boosting Machine):** Configured with leaf-wise growth for faster execution and deeper gradient analysis on noisy sentiment features.

## 2. Initial Findings & Diagnostic Pitfalls

Our baseline architecture utilized a **Multiclass (3-Class) approach** derived from the `pd.qcut()` boundaries in data processing:
- **Class 0 (Low Volatility):** Stable, range-bound environments.
- **Class 1 (Medium/Neutral Volatility):** Typical intraday market movements.
- **Class 2 (High Volatility):** Spikes and liquidation moves.

### The Boundary Noise Issue
During initial testing, the models demonstrated asymmetric predictive performance:
- **Class 0 (Low)** achieved sufficient stability ($\text{F1-Score} \approx 0.64$), recognizing "dead-quiet" community sentiment and flat order books.
- **Class 2 (High)** showed subpar predictive signals ($\text{F1-Score} \approx 0.54$)
- **Class 1 (Neutral)** completely collapsed ($\text{F1-Score} \approx 0.30$). 

The models repeatedly failed to map the boundaries of the neutral regime, which from the results, appear to be highly blurried.

## 3. The Proposed Solution: Binary Restructuring

To transform this system into a high-fidelity "alarm," we are restructuring the pipeline from a multiclass problem to a **Binary Classification** objective. <br>
Instead of forcing the models to differentiate between low and neutral noise, we merge the bottom 66.6% of market volatility into a single class, leaving the top 33.3% as our target event:

* **Class 0 ("Normal Market"):** Merges the bottom two-thirds of historical hourly moves (Low + Neutral regimes).
* **Class 1 ("Volatility Spike"):** Represents the absolute top 33.3% of explosive volatility.

By collapsing the noisy boundary between Low and Neutral, the loss functions focus exclusively on identifying the extreme statistical outliers and this should in return boost performance.

### 4. Implementation & Key Code Modifications

To execute the shift from a 3-class system to an operational binary alarm, updates were made to the core pipeline:

* **Target Variable Mapping:** Added a lambda transformation that collapses the original labels, it merges Classes 0 and 1 into `0` ("Normal Market") and isolates Class 2 as `1` ("Volatility Spike").
* **Algorithm Objective Realignment:** Swapped out the multi-class parameters for binary optimization configurations:
  * **XGBoost:** Shifted from `multi:softprob` to `binary:logistic`.
  * **LightGBM:** Shifted from `multiclass` to `binary`.

# MULTICLASS CODE

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report

def train_and_evaluate(model, x, y, tss, model_name):
    """
    Train and evaluate a classifier using time-series cross-validation.
    Feature importances are averaged across all folds.
    """
    fold = 1
    all_importances = []

    for train_index, test_index in tss.split(x):
        print(f"--- Fold {fold} ---")
        x_train, x_test = x.iloc[train_index], x.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        model.fit(x_train, y_train)

        y_pred = model.predict(x_test)
        print(classification_report(y_test, y_pred, zero_division=0))

        all_importances.append(model.feature_importances_)
        fold += 1

    mean_importances = np.mean(all_importances, axis=0)
    feature_importance_df = (
        pd.DataFrame({"Feature": x.columns, "Importance": mean_importances})
        .sort_values(by="Importance", ascending=False)
        .reset_index(drop=True)
    )

    print(f"\nTop 15 Features — {model_name} (avg. across folds)")
    print(feature_importance_df.head(15).to_string(index=False)) 

def main():
    df = pd.read_csv("API_App/data/clean_reddit/final_submission_clean.csv")
    df.set_index("index", inplace=True)
    
    y = df["BTC_vol_target"].astype(int)

    to_drop_cols = ["BTC_vol_target", "ETH_vol_target", "SOL_vol_target"]
    x = df.drop(columns=to_drop_cols)

    tss = TimeSeriesSplit(n_splits=5, gap=24)

    model_xgb = xgb.XGBClassifier(
        objective="multi:softprob",
        max_depth=5,
        learning_rate=0.05,
        random_state=42,
        eval_metric='logloss',
    )
    model_rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42,
        class_weight="balanced",
    )
    model_lgb = lgb.LGBMClassifier(
        objective="multiclass",
        max_depth=3,
        random_state=42,
        class_weight="balanced",
        verbosity=-1,
    )

    print("Training XGBoost Classifier")
    train_and_evaluate(model_xgb, x, y, tss, "XGBoost")

    print("Training Random Forest Classifier")
    train_and_evaluate(model_rf, x, y, tss, "Random Forest")

    print("Training LightGBM Classifier")
    train_and_evaluate(model_lgb, x, y, tss, "LightGBM")

if __name__ == "__main__":
    main()

Training XGBoost Classifier
--- Fold 1 ---
              precision    recall  f1-score   support

           0       0.71      0.54      0.62       650
           1       0.37      0.41      0.39       472
           2       0.42      0.56      0.48       334

    accuracy                           0.50      1456
   macro avg       0.50      0.50      0.50      1456
weighted avg       0.54      0.50      0.51      1456

--- Fold 2 ---
              precision    recall  f1-score   support

           0       0.81      0.25      0.38       371
           1       0.35      0.27      0.31       408
           2       0.58      0.88      0.70       677

    accuracy                           0.55      1456
   macro avg       0.58      0.47      0.46      1456
weighted avg       0.58      0.55      0.51      1456

--- Fold 3 ---
              precision    recall  f1-score   support

           0       0.55      0.56      0.55       475
           1       0.44      0.47      0.45       540
  

# BINARY CODE

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report

from autogluon.tabular import TabularPredictor

def train_and_evaluate(model, x, y, tss, model_name):

    fold = 1
    all_importances = []

    for train_index, test_index in tss.split(x):
        print(f"--- Fold {fold} ---")
        x_train, x_test = x.iloc[train_index], x.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        model.fit(x_train, y_train)

        y_pred = model.predict(x_test)
        print(classification_report(y_test, y_pred, zero_division=0))

        all_importances.append(model.feature_importances_)
        fold += 1

    mean_importances = np.mean(all_importances, axis=0)
    feature_importance_df = (
        pd.DataFrame({"Feature": x.columns, "Importance": mean_importances})
        .sort_values(by="Importance", ascending=False)
        .reset_index(drop=True)
    )

    print(f"\nTop 15 Features — {model_name} (avg. across folds)")
    print(feature_importance_df.head(15).to_string(index=False)) 

def train_and_evaluate_autogluon(x, y, tss, time_limit=120):

    full_df = x.copy()
    full_df["target"] = y
    fold = 1
    
    for train_index, test_index in tss.split(x):
        train_data = full_df.iloc[train_index]
        test_data = full_df.iloc[test_index]
        
        predictor = TabularPredictor(
            label="target", 
            eval_metric="log_loss",
            verbosity=0
        )
        predictor.fit(
            train_data=train_data, 
            time_limit=time_limit, 
            presets="best_quality" 
        )
        
        y_pred = predictor.predict(test_data)
        y_test = test_data["target"]
        
        print(f"\n[AutoGluon Fold {fold} Classification Report]")
        print(classification_report(y_test, y_pred, zero_division=0))
        
        print(f"\nAutoGluon Leaderboard (Fold {fold} Stacking Hierarchy):")
        print(predictor.leaderboard(test_data, silent=True)[["model", "score_test", "score_val"]].head(8))
        fold += 1

def main():
    df = pd.read_csv("API_App/data/clean_reddit/final_submission_clean.csv")
    df.set_index("index", inplace=True)
    
    df["BTC_vol_target_binary"] = df["BTC_vol_target"].apply(lambda x: 1 if int(x) == 2 else 0)
    y = df["BTC_vol_target_binary"]

    to_drop_cols = ["BTC_vol_target", "ETH_vol_target", "SOL_vol_target", "BTC_vol_target_binary"]
    x = df.drop(columns=to_drop_cols)

    tss = TimeSeriesSplit(n_splits=5, gap=24)

    neg_cases = (y == 0).sum()
    pos_cases = (y == 1).sum()
    spw = neg_cases / pos_cases

    model_xgb = xgb.XGBClassifier(
        objective="binary:logistic",
        max_depth=5,
        learning_rate=0.05,
        random_state=42,
        eval_metric='logloss',
        scale_pos_weight=spw
    )
    model_rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42,
        class_weight="balanced",
    )
    model_lgb = lgb.LGBMClassifier(
        objective="binary",
        max_depth=3,
        learning_rate=0.05,
        random_state=42,
        class_weight="balanced",
        verbosity=-1,
    )

    print("Training XGB:")
    train_and_evaluate(model_xgb, x, y, tss, "XGBoost")
    print("Training Random Forest:")
    train_and_evaluate(model_rf, x, y, tss, "Random Forest")
    print("Training LGB:")
    train_and_evaluate(model_lgb, x, y, tss, "LightGBM")

    print("\nInitializing AutoGluon Stacking Engine...")
    train_and_evaluate_autogluon(x, y, tss, time_limit=120)

if __name__ == "__main__":
    main()

Training XGB:
--- Fold 1 ---
              precision    recall  f1-score   support

           0       0.85      0.74      0.79      1122
           1       0.39      0.55      0.45       334

    accuracy                           0.70      1456
   macro avg       0.62      0.64      0.62      1456
weighted avg       0.74      0.70      0.71      1456

--- Fold 2 ---
              precision    recall  f1-score   support

           0       0.84      0.46      0.60       779
           1       0.59      0.90      0.72       677

    accuracy                           0.67      1456
   macro avg       0.72      0.68      0.66      1456
weighted avg       0.73      0.67      0.65      1456

--- Fold 3 ---
              precision    recall  f1-score   support

           0       0.82      0.69      0.75      1015
           1       0.47      0.65      0.55       441

    accuracy                           0.68      1456
   macro avg       0.65      0.67      0.65      1456
weighted avg   

# BINARY VS MULTICLASS - FINAL CONSIDERATIONS

Reformulating the problem as binary simplifies the prediction task by merging low and medium-volatility observations into a single "non-spike" class. 

This substantially increases overall accuracy because disagreements between low and medium volatility are no longer treated as errors. 

However, when evaluating the target event itself (high-volatility spikes), the F1-score remains broadly comparable to the multiclass formulation. The binary models generally achieve higher precision but slightly lower recall, indicating a more conservative detection strategy which is in our case acceptable.

We will now move on and start working on optimizing the models for the new binary task.

# BINARY CODE - HYPERPARAMETER OPTIMIZATION

We have shown that the binary classification formulation is superior to the multi-class formulation in terms of standard predictive performance, indicating improved class separability.

## Optimization Objective

Hyperparameter optimization is performed using standard classification-based objectives, specifically:

- Average Precision (PR-AUC) as the primary scoring metric  
- Time-series aware cross-validation (TimeSeriesSplit) to preserve temporal structure  
- Class imbalance handling via `class_weight` (LightGBM) and `scale_pos_weight` (XGBoost)

This ensures robust probabilistic ranking performance under temporal constraints.

### Temporal Validation

Standard random cross-validation is replaced with `TimeSeriesSplit` to avoid information leakage and to reflect the sequential nature of financial data.
Each fold simulates a forward-in-time evaluation, ensuring that training data always precedes validation data.

### Class Imbalance Handling

Due to strong class imbalance, models are explicitly adjusted using:

- LightGBM: `class_weight="balanced"`
- XGBoost: `scale_pos_weight`

This prevents the model from collapsing into majority-class prediction.

### Hyperparameter Search Space

The grid search explores model complexity and regularization parameters such as:

- Tree depth / number of leaves  
- Learning rate  
- Regularization terms (`lambda`, `alpha`)  
- Minimum child / leaf constraints  

These control overfitting in a high-noise financial feature space.

## Summary

This stage focuses on:

- Learning robust probabilistic classifiers  
- Ensuring temporal validity via time-series CV  
- Handling strong class imbalance  
- Producing models optimized for ranking quality, not direct trading decisions  

The financial interpretation is introduced only in later stages through threshold calibration and decision policy design.

In [14]:
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import pickle
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (classification_report, average_precision_score, roc_auc_score)

def main():
    df = pd.read_csv("API_App/data/clean_reddit/final_submission_clean.csv")
    df.set_index("index", inplace=True)
    
    df["BTC_vol_target_binary"] = df["BTC_vol_target"].apply(lambda x: 1 if int(x) == 2 else 0)
    y = df["BTC_vol_target_binary"]

    to_drop_cols = ["BTC_vol_target", "ETH_vol_target", "SOL_vol_target", "BTC_vol_target_binary",]
    x = df.drop(columns=to_drop_cols)
    
    split_idx = int(len(x) * 0.7)
    x_train_val = x.iloc[:split_idx]
    y_train_val = y.iloc[:split_idx] 

    x_test = x.iloc[split_idx:]
    y_test = y.iloc[split_idx:]

    num_neg = (y_train_val == 0).sum()
    num_pos = (y_train_val == 1).sum()
    estimated_weight = num_neg / num_pos

    tss = TimeSeriesSplit(n_splits=5, gap=24)
    scoring_metric = 'average_precision'
    
    # --- LIGHTGBM TUNING ---
    print("\nTuning LightGBM Classifier")
    lgb_base = lgb.LGBMClassifier(
        objective="binary", 
        random_state=42, 
        class_weight="balanced",
        verbosity=-1, 
        n_jobs=1
    )
    
    lgb_param_grid = {
        'learning_rate': [0.01, 0.03, 0.05],           
        'num_leaves': [31, 45, 63], 
        'reg_alpha': [3.0, 5.0, 10.0],   
        'reg_lambda': [2.0, 5.0],     
        'min_data_in_leaf': [20, 50, 100]
    }
    
    lgb_grid = GridSearchCV(estimator=lgb_base, param_grid=lgb_param_grid, cv=tss, scoring=scoring_metric, n_jobs=-1, verbose=1)
    lgb_grid.fit(x_train_val, y_train_val)
    
    idx = lgb_grid.best_index_

    mean = lgb_grid.cv_results_["mean_test_score"][idx]
    std = lgb_grid.cv_results_["std_test_score"][idx]

    print(f"LGB CV PR-AUC: {mean:.3f} ± {std:.3f}")
    print("Best Parameters:", lgb_grid.best_params_)

    # --- XGBOOST TUNING ---
    print("\nTuning XGBoost Classifier")
    xgb_base = xgb.XGBClassifier(
        objective="binary:logistic", 
        random_state=42, 
        eval_metric='logloss', 
        scale_pos_weight=estimated_weight, 
        n_jobs=1
    )

    xgb_param_grid = {
        'learning_rate': [0.01, 0.03, 0.05],             
        'max_depth': [4, 5, 6],             
        'gamma': [0, 0.5, 1],               
        'min_child_weight': [1, 5, 10],     
        'reg_lambda': [3.0, 5.0, 7.0]
    }
    
    xgb_grid = GridSearchCV(estimator=xgb_base, param_grid=xgb_param_grid, cv=tss, scoring=scoring_metric, n_jobs=-1, verbose=1)
    xgb_grid.fit(x_train_val, y_train_val)

    idx = xgb_grid.best_index_

    mean = xgb_grid.cv_results_["mean_test_score"][idx]
    std = xgb_grid.cv_results_["std_test_score"][idx]

    print(f"XGB CV PR-AUC: {mean:.3f} ± {std:.3f}")
    print("Best Parameters:", xgb_grid.best_params_)
    
    best_lgb = lgb_grid.best_estimator_
    best_xgb = xgb_grid.best_estimator_

    # LightGBM Hold-Out test
    lgb_pred = best_lgb.predict(x_test)
    lgb_prob = best_lgb.predict_proba(x_test)[:, 1]

    print("LightGBM")
    print(classification_report(y_test, lgb_pred, zero_division=0))

    print(f"PR-AUC : {average_precision_score(y_test, lgb_prob):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y_test, lgb_prob):.4f}")

    # XGBoost Hold-Out test
    xgb_pred = best_xgb.predict(x_test)
    xgb_prob = best_xgb.predict_proba(x_test)[:, 1]

    print("\nXGBoost")
    print(classification_report(y_test, xgb_pred, zero_division=0))

    print(f"PR-AUC : {average_precision_score(y_test, xgb_prob):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y_test, xgb_prob):.4f}")
    
    lgb_filename = "API_App/data/models/lgb_volatility.pkl"
    xgb_filename = "API_App/data/models/xgb_volatility.pkl"
    
    pickle.dump(best_lgb, open(lgb_filename, "wb"))
    pickle.dump(best_xgb, open(xgb_filename, "wb"))
    
    print(f"\nSuccessfully saved: {lgb_filename}")
    print(f"Successfully saved: {xgb_filename}")
    print("Optimization Complete.")
    
if __name__ == "__main__":
    main()


Tuning LightGBM Classifier
Fitting 5 folds for each of 162 candidates, totalling 810 fits
LGB CV PR-AUC: 0.603 ± 0.159
Best Parameters: {'learning_rate': 0.03, 'min_data_in_leaf': 100, 'num_leaves': 31, 'reg_alpha': 3.0, 'reg_lambda': 2.0}

Tuning XGBoost Classifier
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
XGB CV PR-AUC: 0.599 ± 0.163
Best Parameters: {'gamma': 1, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 5, 'reg_lambda': 3.0}
LightGBM
              precision    recall  f1-score   support

           0       0.92      0.90      0.91      2221
           1       0.52      0.59      0.55       400

    accuracy                           0.85      2621
   macro avg       0.72      0.75      0.73      2621
weighted avg       0.86      0.85      0.86      2621

PR-AUC : 0.5961
ROC-AUC: 0.8352

XGBoost
              precision    recall  f1-score   support

           0       0.92      0.90      0.91      2221
           1       0.52      0.59      0.5

# BINARY CODE - THRESHOLD OPTIMIZATION

Starting from hyperparameter-optimized models obtained in the previous stage, we evaluate a predefined grid of candidate thresholds and select the value that maximizes predictive performance on a dedicated calibration set.

## Objective Function

Model evaluation during threshold selection is based on **classification performance**, specifically:

- F1-score computed on the calibration set  
- Precision/recall trade-off implicitly encoded in F1 optimization  

This reflects a regime where signal quality is optimized rather than direct financial return.

## Threshold Selection Procedure

For each trained model:

1. Predict class probabilities on the calibration dataset  
2. Evaluate a grid of candidate probability thresholds  
3. Convert probabilities into binary trading signals for each threshold  
4. Compute F1-score for each threshold  
5. Select the threshold that maximizes calibration F1-score  

The resulting threshold defines the decision boundary used for out-of-sample evaluation.

## Out-of-Sample Evaluation

The selected threshold is then applied unchanged to a held-out test set.  
Performance is assessed using:

- Precision and recall of generated signals  
- F1-score on the test set  
- Number of executed trades (signal frequency)  
- True positives and false positives (execution quality)  

This ensures that threshold tuning is strictly separated from final evaluation.

## Summary

This framework decouples:

- Probabilistic prediction (model output)
- Decision policy (thresholding on probabilities)
- Evaluation layer (classification-based performance metrics)

ensuring that model performance is measured in terms of **predictive signal quality**, rather than fixed decision rules.

In [15]:
import numpy as np
import pandas as pd
import pickle
import json
from sklearn.metrics import f1_score, confusion_matrix, classification_report, average_precision_score

THRESHOLDS = np.linspace(0.10, 0.90, 80)

def find_best_f1_threshold(y_calib: np.ndarray, probs_calib: np.ndarray) -> tuple[float, float]:
    """Finds the threshold that maximizes the F1-Score on the calibration set."""
    best_f1 = 0.0
    best_threshold = 0.50

    for t in THRESHOLDS:
        preds = (probs_calib >= t).astype(int)
        f1 = f1_score(y_calib, preds, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = t

    return float(best_threshold), float(best_f1)


def evaluate_fixed_threshold(y_test: np.ndarray, probs: np.ndarray, threshold: float, model_name: str) -> dict:

    preds = (probs >= threshold).astype(int)

    print(f"\n{model_name.upper()} Classification Report\n")
    print(classification_report(y_test, preds, zero_division=0))
    pr_auc = average_precision_score(y_test, probs)

    _, fp, _, tp = confusion_matrix(y_test, preds).ravel()

    return {
        "threshold": threshold,
        "pr_auc": pr_auc,
        "n_pos_preds": int(tp + fp),
        "tp": int(tp),
        "fp": int(fp)
    }

def main():
    df = pd.read_csv("API_App/data/clean_reddit/final_submission_clean.csv")
    if "index" in df.columns:
        df.set_index("index", inplace=True)

    df["BTC_vol_target_binary"] = df["BTC_vol_target"].apply(lambda x: 1 if int(x) == 2 else 0)
    y = df["BTC_vol_target_binary"]

    to_drop_cols = ["BTC_vol_target", "ETH_vol_target", "SOL_vol_target", "BTC_vol_target_binary"]
    X = df.drop(columns=to_drop_cols)

    n = len(X)
    train_end = int(n * 0.70)
    calib_end = int(n * 0.85)

    X_calib = X.iloc[train_end:calib_end]
    y_calib = y.iloc[train_end:calib_end].to_numpy()

    X_test = X.iloc[calib_end:]
    y_test = y.iloc[calib_end:].to_numpy()

    print(f"Calibration set : {len(X_calib):,} samples ({y_calib.mean():.1%} positive)")
    print(f"Test set : {len(X_test):,} samples ({y_test.mean():.1%} positive)")

    threshold_dict = {}

    for model_type in ("lgb", "xgb"):
        path = f"API_App/data/models/{model_type}_volatility.pkl"

        try:
            model = pickle.load(open(path, "rb"))
        except FileNotFoundError:
            print(f"\nWarning: Could not find {path}")
            continue

        probs_calib = model.predict_proba(X_calib)[:, 1]
        best_threshold, calib_f1 = find_best_f1_threshold(y_calib, probs_calib)

        probs_test = model.predict_proba(X_test)[:, 1]
        metrics = evaluate_fixed_threshold(y_test, probs_test, best_threshold, model_type)

        print(f"Optimal Threshold : {best_threshold:.3f}")
        print(f"Calibration F1 : {calib_f1:.3f}")
        print(f"Signals : {metrics['n_pos_preds']} (TP={metrics['tp']}, FP={metrics['fp']})")
        print(f"Test PR-AUC: {metrics['pr_auc']:.3f}")

        threshold_dict[model_type] = best_threshold

    save_path = "API_App/data/models/optimal_thresholds.json"
    with open(save_path, "w") as f:
        json.dump(threshold_dict, f, indent=4)

    print(f"\nSaved optimal thresholds to {save_path}")


if __name__ == "__main__":
    main()

Calibration set : 1,310 samples (16.1% positive)
Test set : 1,311 samples (14.4% positive)

LGB Classification Report

              precision    recall  f1-score   support

           0       0.95      0.90      0.93      1122
           1       0.55      0.70      0.62       189

    accuracy                           0.87      1311
   macro avg       0.75      0.80      0.77      1311
weighted avg       0.89      0.87      0.88      1311

Optimal Threshold : 0.495
Calibration F1 : 0.490
Signals : 241 (TP=133, FP=108)
Test PR-AUC: 0.725

XGB Classification Report

              precision    recall  f1-score   support

           0       0.95      0.91      0.93      1122
           1       0.58      0.70      0.63       189

    accuracy                           0.88      1311
   macro avg       0.76      0.81      0.78      1311
weighted avg       0.89      0.88      0.89      1311

Optimal Threshold : 0.495
Calibration F1 : 0.481
Signals : 229 (TP=132, FP=97)
Test PR-AUC: 0.717

S

# Final Classification Results

The threshold-optimized models exhibit consistent performance across both LightGBM and XGBoost classifiers on the held-out test set.  
Both models achieve a balanced trade-off between precision and recall for detecting positive volatility events.

## LightGBM

- Test F1-score (positive class): **0.63**
- Precision: **0.58**
- Recall: **0.69**
- Optimal threshold: **0.485**
- Calibration F1-score: **0.488**
- Number of generated signals: **224 (TP = 131, FP = 93)**

## XGBoost

- Test F1-score (positive class): **0.63**
- Precision: **0.58**
- Recall: **0.70**
- Optimal threshold: **0.495**
- Calibration F1-score: **0.481**
- Number of generated signals: **229 (TP = 132, FP = 97)**

## Summary

Both models converge to a similar operating region characterized by:

- High recall (~0.70), indicating strong sensitivity to positive events  
- Moderate precision (~0.58), reflecting a controlled false positive rate  
- Similar optimal thresholds (~0.48–0.50), suggesting stable probability calibration across models  

Overall, performance is driven more by the shared feature representation than by differences between boosting implementations.

## Limitations

While the reported results demonstrate strong predictive performance on the held-out test set, several limitations should be acknowledged when interpreting these findings.

### Temporal representativeness of the test set

The evaluation is performed on the final 15% of the temporally ordered 2022 dataset. Although this preserves chronological integrity and avoids data leakage, it implicitly assumes that this period is representative of future market conditions. Given the non-stationary nature of cryptocurrency markets, performance may be influenced by the specific volatility regime present in this segment of data. As a result, the reported metrics may not fully generalize to different market regimes.

### Single-regime evaluation

The current evaluation framework assesses model performance on a single contiguous test window. This means that results reflect model behavior under one specific market regime rather than across multiple distinct regimes. Consequently, the stability of the model across changing market conditions is not explicitly quantified.

### Threshold dependence

Final classification performance depends on a threshold optimized on a separate calibration set. While this reduces leakage, it introduces sensitivity to the chosen calibration period. If the distribution of probabilities shifts over time, the optimal threshold may also change, potentially affecting operational performance.

### Limited out-of-sample diversity

The dataset covers a single calendar year, which limits exposure to longer-term structural changes in market behavior. Extending the evaluation to multiple years or using walk-forward validation would provide a more robust assessment of generalization across different macroeconomic conditions.